In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

In [2]:
# -------------------------------------------------------------------
# Setup & Getting the Data
# -------------------------------------------------------------------
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# Missing values handling
categorical = list(df.select_dtypes(include=['object', 'category']).columns)
numerical = list(df.select_dtypes(include=['number']).columns)

target_col = 'converted'
if target_col in numerical:
    numerical.remove(target_col)

for col in categorical:
    df[col] = df[col].fillna('NA')

for col in numerical:
    df[col] = df[col].fillna(0.0)

C:\Users\saulp\AppData\Local\Temp\ipykernel_81772\3534035258.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = list(df.select_dtypes(include=['object', 'category']).columns)


In [4]:
# Exact split specified in instructions
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

del df_train[target_col]
del df_val[target_col]
del df_test[target_col]

cat_features = [col for col in categorical if col != target_col]
all_features = numerical + cat_features

In [5]:
# -------------------------------------------------------------------
# Question 1: ROC AUC Feature Importance
# -------------------------------------------------------------------
q1_features = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
auc_scores = {}

for col in q1_features:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    auc_scores[col] = auc

highest_auc_feature = max(auc_scores, key=auc_scores.get)
print("Q1. ROC AUC for numerical features:")
for col, score in auc_scores.items():
    print(f"    {col}: {score:.4f}")
print(f"    Highest AUC feature: {highest_auc_feature}\n")

Q1. ROC AUC for numerical features:
    lead_score: 0.7882
    number_of_courses_viewed: 0.7230
    interaction_count: 0.7649
    annual_income: 0.6082
    Highest AUC feature: lead_score



In [6]:
# -------------------------------------------------------------------
# Question 2: Training Logistic Regression Model & Validation AUC
# -------------------------------------------------------------------
train_dicts = df_train[all_features].to_dict(orient='records')
val_dicts = df_val[all_features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred_val = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred_val)
print(f"Q2. Validation AUC: {round(val_auc, 3)} (exact: {val_auc:.4f})\n")

Q2. Validation AUC: 0.732 (exact: 0.7318)



In [7]:
# -------------------------------------------------------------------
# Question 3: Precision and Recall Intersection
# -------------------------------------------------------------------
thresholds = np.arange(0.0, 1.01, 0.01)
diffs = {}

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)

    predict_positive = (y_pred_val >= t)

    tp = (predict_positive & actual_positive).sum()
    fp = (predict_positive & actual_negative).sum()
    fn = (~predict_positive & actual_positive).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0

    if p > 0 and r > 0:
        diffs[t] = abs(p - r)

intersect_threshold = min(diffs, key=diffs.get)
print(f"Q3. Precision & Recall intersect threshold: {round(intersect_threshold, 2)}\n")

Q3. Precision & Recall intersect threshold: 0.63



In [8]:
# -------------------------------------------------------------------
# Question 4: F1 Score
# -------------------------------------------------------------------
f1_scores = {}

for t in thresholds:
    predict_positive = (y_pred_val >= t)
    
    tp = (predict_positive & (y_val == 1)).sum()
    fp = (predict_positive & (y_val == 0)).sum()
    fn = (~predict_positive & (y_val == 1)).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    f1_scores[t] = f1

max_f1_threshold = max(f1_scores, key=f1_scores.get)
print(f"Q4. Threshold with maximal F1 score: {round(max_f1_threshold, 2)}\n")

Q4. Threshold with maximal F1 score: 0.41



In [9]:
# -------------------------------------------------------------------
# Question 5: 5-Fold Cross Validation
# -------------------------------------------------------------------
def train_and_eval(df_tr, y_tr, df_v, y_v, C=1.0):
    dicts_tr = df_tr[all_features].to_dict(orient='records')
    dicts_v = df_v[all_features].to_dict(orient='records')

    dv_fold = DictVectorizer(sparse=False)
    X_tr = dv_fold.fit_transform(dicts_tr)
    X_v = dv_fold.transform(dicts_v)

    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    m.fit(X_tr, y_tr)

    y_pred = m.predict_proba(X_v)[:, 1]
    return roc_auc_score(y_v, y_pred)

kf = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for train_idx, val_idx in kf.split(df_full_train):
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    y_fold_train = df_fold_train[target_col].values
    y_fold_val = df_fold_val[target_col].values

    score = train_and_eval(df_fold_train, y_fold_train, df_fold_val, y_fold_val, C=1.0)
    scores.append(score)

std_score = np.std(scores)
print(f"Q5. Standard deviation of scores across folds: {round(std_score, 3)} (exact: {std_score:.4f})\n")

Q5. Standard deviation of scores across folds: 0.007 (exact: 0.0069)



In [10]:
# -------------------------------------------------------------------
# Question 6: Hyperparameter Tuning C
# -------------------------------------------------------------------
c_values = [0.000001, 0.001, 1]
print("Q6. Cross-validation parameter C evaluation:")

for C in c_values:
    c_scores = []
    for train_idx, val_idx in kf.split(df_full_train):
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        y_fold_train = df_fold_train[target_col].values
        y_fold_val = df_fold_val[target_col].values

        score = train_and_eval(df_fold_train, y_fold_train, df_fold_val, y_fold_val, C=C)
        c_scores.append(score)

    mean_auc = np.mean(c_scores)
    std_auc = np.std(c_scores)
    print(f"    C={C:<8} -> Mean ROC AUC: {round(mean_auc, 3):.3f}, Std: {round(std_auc, 3):.3f}")

Q6. Cross-validation parameter C evaluation:
    C=1e-06    -> Mean ROC AUC: 0.617, Std: 0.019
    C=0.001    -> Mean ROC AUC: 0.749, Std: 0.010
    C=1        -> Mean ROC AUC: 0.732, Std: 0.007
